# Phase 1: Data Preprocessing for Olympic Games
Run these cells one by one (`Shift + Enter`) to see how data transforms step by step. This matches your exact problem statement for creating athlete participation counts, medal totals, and country-wise performance across years!

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

csv_path = '/Users/raj/Downloads/athlete_events.csv'

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
print("Loading data...")
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print("First 5 rows of the raw data (One row = One Athlete in one Event):")
df.head()

In [ ]:
print("Missing values in Medal column before cleaning:", df['Medal'].isnull().sum())

df['Medal'] = df['Medal'].fillna('None')

print("Missing values after cleaning:", df['Medal'].isnull().sum())
df[['Name', 'Event', 'Medal']].head(10)

In [ ]:
medal_dummies = pd.get_dummies(df['Medal'], prefix='Medal', dtype=int)

if 'Medal_None' in medal_dummies.columns:
    medal_dummies = medal_dummies.drop('Medal_None', axis=1)

df = pd.concat([df, medal_dummies], axis=1)

print("Notice the 3 new Medal binary columns added:")
df[['Name', 'Medal', 'Medal_Bronze', 'Medal_Silver', 'Medal_Gold']].head(10)

In [ ]:

country_year_data = df.groupby(['NOC', 'Year']).agg(
    Athlete_Participation_Count=('ID', 'nunique'),
    Event_Count=('Event', 'nunique'),
    Total_Gold=('Medal_Gold', 'sum'),
    Total_Silver=('Medal_Silver', 'sum'),
    Total_Bronze=('Medal_Bronze', 'sum')
).reset_index()

country_year_data['Total_Medals'] = (
    country_year_data['Total_Gold'] + 
    country_year_data['Total_Silver'] + 
    country_year_data['Total_Bronze']
)

print(f"New Aggregated Dataset Shape: {country_year_data.shape}")
print("Every row represents ONE country's overall performance in ONE specific year!")
country_year_data.head(10)

In [ ]:

scaler = StandardScaler()

numeric_columns = ['Athlete_Participation_Count', 'Event_Count']
country_year_data_scaled = country_year_data.copy()
country_year_data_scaled[numeric_columns] = scaler.fit_transform(country_year_data_scaled[numeric_columns])

print("Final Normalized Data for Machine Learning (Phase 1 Complete!):")
country_year_data_scaled.head(10)

In [ ]:
features = ['Athlete_Participation_Count', 'Event_Count', 'Total_Gold', 'Total_Silver', 'Total_Bronze']
X = country_year_data_scaled[features]
y = country_year_data_scaled['Total_Medals']

selector = SelectKBest(score_func=f_regression, k=3)
selector.fit(X, y)

scores = pd.DataFrame({'Feature': features, 'Score': selector.scores_})
scores = scores.sort_values('Score', ascending=False).reset_index(drop=True)

selected_features = scores['Feature'].head(3).tolist()
X_selected = selector.transform(X)

print('Feature Scores:')
print(scores)
print('\nSelected Features:', selected_features)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

features = ['Athlete_Participation_Count', 'Event_Count']
y = country_year_data['Total_Medals']

X_train, X_test, y_train, y_test = train_test_split(country_year_data[features], y, test_size=0.2, random_state=42)
model1 = KNeighborsRegressor().fit(X_train, y_train)
acc1 = model1.score(X_test, y_test)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(country_year_data_scaled[features], y, test_size=0.2, random_state=42)
model2 = KNeighborsRegressor().fit(X_train_s, y_train_s)
acc2 = model2.score(X_test_s, y_test_s)

print(f"Accuracy before Normalization: {acc1:.4f}")
print(f"Accuracy after Normalization: {acc2:.4f}")